# Problem 4: Seller risk score

There are about 3,000 sellers and the top 10% bring roughly two-thirds of GMV. The seller success team is small, so they need to know which sellers to focus on. I also want to check that the score predicts future problems, not just past ones.

Steps:
1. Start from the seller scorecard made in SQL (`sql/04_seller_scorecard.sql`)
2. Shrink the rates towards the platform average (empirical Bayes), so a seller with 3 orders and 1 late isn't treated as "33% late"
3. Combine late handoff, bad reviews and cancellations into one score from 0 to 100
4. Put sellers into four groups using GMV and risk score
5. Back-test: build the score on data up to Mar 2018 and check how the sellers did in Apr-Aug 2018

In [1]:
import sys, sqlite3, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import chart_style as cs
cs.apply(); warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.2f}".format)
con = sqlite3.connect(ROOT / "data" / "olist.db")
OUT = ROOT / "data" / "processed"

## 1. GMV concentration

In [2]:
par = pd.read_csv(OUT / "seller_pareto.csv")
share_at = lambda pct: par.loc[par.pct_of_sellers <= pct, "cum_pct_of_gmv"].max()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(par.pct_of_sellers, par.cum_pct_of_gmv, color=cs.PLUM)
ax.fill_between(par.pct_of_sellers, par.cum_pct_of_gmv, color=cs.PLUM, alpha=0.1)
for pct in (10, 20):
    v = share_at(pct)
    ax.scatter([pct], [v], color=cs.PLUM, s=50, zorder=3, edgecolor="white", linewidth=2)
    ax.annotate(f"Top {pct}% of sellers = {v:.0f}% of GMV", (pct, v), xytext=(12, -14), textcoords="offset points", fontsize=9, color=cs.DARK)
ax.set_xlabel("% of sellers (ranked by GMV)"); ax.set_ylabel("cumulative % of GMV"); ax.set_xlim(0, 100); ax.set_ylim(0, 102)
ax.set_title("Seller GMV concentration")
cs.save(fig, "p4_seller_pareto")

PosixPath('/home/claude/olist-ops-growth-case-study/images/p4_seller_pareto.png')

## 2. Risk score

In [3]:
def score_sellers(orders: pd.DataFrame, cancels: pd.DataFrame, min_orders: int = 20, m: float = 20.0) -> pd.DataFrame:
    """orders: delivered orders (seller_id, seller_late_handoff, is_late, review_score, items_value)
       cancels: canceled orders (seller_id). Returns one row per seller with shrunk rates and a 0-100 risk score."""
    g = orders.groupby("seller_id").agg(
        delivered=("order_id", "count"), gmv=("items_value", "sum"),
        late_handoff=("seller_late_handoff", "sum"), late=("is_late", "sum"),
        reviews=("review_score", "count"), bad=("review_score", lambda s: (s <= 2).sum()))
    g["canceled"] = cancels.groupby("seller_id").size().reindex(g.index, fill_value=0)
    g = g[g.delivered >= min_orders].copy()
    # platform priors
    p_hand = orders.seller_late_handoff.mean(); p_bad = (orders.review_score <= 2).sum() / orders.review_score.notna().sum()
    p_can = len(cancels) / (len(orders) + len(cancels))
    # empirical-Bayes (beta prior with weight m pseudo-orders)
    g["handoff_rate"] = (g.late_handoff + m * p_hand) / (g.delivered + m)
    g["bad_review_rate"] = (g.bad + m * p_bad) / (g.reviews + m)
    g["cancel_rate"] = (g.canceled + m * p_can) / (g.delivered + g.canceled + m)
    W = {"handoff_rate": 0.40, "bad_review_rate": 0.40, "cancel_rate": 0.20}
    g["risk_score"] = sum(g[k].rank(pct=True) * w for k, w in W.items()) * 100
    return g

# note: SELECT * is much faster here than selecting a few columns (SQLite plans the view differently)
d = pd.read_sql("SELECT * FROM v_delivered", con)[["order_id", "seller_id", "seller_late_handoff", "is_late",
                                                     "review_score", "items_value", "purchase_month"]]
# canceled orders with their seller (merged in pandas, joining the window-function view in SQLite was slow)
canc = (pd.read_sql("""SELECT order_id, SUBSTR(order_purchase_timestamp,1,7) AS purchase_month
                       FROM orders WHERE order_status = 'canceled'""", con)
          .merge(pd.read_sql("SELECT order_id, seller_id FROM v_primary_seller", con), on="order_id"))
sc = score_sellers(d, canc)
print(f"Sellers scored (>= 20 delivered orders): {len(sc):,} of {d.seller_id.nunique():,} | they carry {sc.gmv.sum()/d.items_value.sum():.0%} of GMV")
sc[["delivered", "gmv", "handoff_rate", "bad_review_rate", "cancel_rate", "risk_score"]].describe().round(3)

Sellers scored (>= 20 delivered orders): 795 of 2,959 | they carry 82% of GMV


,delivered,gmv,handoff_rate,bad_review_rate,cancel_rate,risk_score
count,795.00,795.00,795.00,795.00,795.00,795.00
mean,106.37,"13,681.33",0.09,0.12,0.00,50.06
std,181.61,"24,406.64",0.08,0.05,0.01,20.43
min,20.00,230.74,0.01,0.03,0.00,2.44
25%,31.00,"3,552.32",0.04,0.09,0.00,34.76
50%,53.00,"6,406.23",0.06,0.11,0.00,49.56
75%,104.00,"12,861.91",0.11,0.15,0.00,66.35
max,"1,801.00","225,000.34",0.56,0.52,0.05,97.87


Without shrinkage, the top of the list would be filled with tiny sellers who had one bad order. With it, a seller needs enough orders behind the numbers to rank high. Weights are 40% late handoff, 40% bad reviews, 20% cancellations. I gave less weight to cancellations because customers feel the other two more directly. The weights are easy to change in the function.

## 3. Quadrants

In [4]:
gmv_cut = sc.gmv.quantile(0.75); risk_cut = 70
def quad(r_):
    hi_v, hi_r = r_.gmv >= gmv_cut, r_.risk_score >= risk_cut
    return ("Q1 Fix now (AM-owned)" if hi_v and hi_r else "Q2 Protect (keep happy)" if hi_v
            else "Q3 Automate nudges / restrict" if hi_r else "Q4 Self-serve")
sc["quadrant"] = sc.apply(quad, axis=1)

colors = {"Q1 Fix now (AM-owned)": cs.BRICK, "Q2 Protect (keep happy)": cs.GREEN,
          "Q3 Automate nudges / restrict": cs.MUSTARD, "Q4 Self-serve": "#c8c8c8"}
fig, ax = plt.subplots(figsize=(10, 5.6))
for q, c in colors.items():
    s_ = sc[sc.quadrant == q]
    ax.scatter(s_.gmv, s_.risk_score, s=np.clip(s_.delivered / 6, 10, 160), color=c, alpha=0.75,
               edgecolor="white", linewidth=0.8, label=f"{q} ({len(s_)})")
ax.set_xscale("log"); ax.axvline(gmv_cut, color=cs.GREY, ls="--", lw=1); ax.axhline(risk_cut, color=cs.GREY, ls="--", lw=1)
from matplotlib.ticker import FuncFormatter
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"R$ {v/1000:,.0f}k" if v >= 1000 else f"R$ {v:,.0f}"))
ax.set_xlabel("Seller GMV, Jan-17 to Aug-18 (log scale)"); ax.set_ylabel("Risk score (0-100)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2)
q1 = sc[sc.quadrant.str.startswith("Q1")]
ax.set_title("Seller priority matrix (GMV vs risk score)")
cs.save(fig, "p4_seller_risk_matrix")

qs = sc.groupby("quadrant").agg(sellers=("gmv", "size"), gmv=("gmv", "sum"), late_handoffs=("late_handoff", "sum"),
                                 bad_reviews=("bad", "sum"), avg_risk=("risk_score", "mean"))
qs["gmv_share"] = qs.gmv / qs.gmv.sum(); qs["late_handoff_share"] = qs.late_handoffs / qs.late_handoffs.sum()
qs.round(3)

,sellers,gmv,late_handoffs,bad_reviews,avg_risk,gmv_share,late_handoff_share
quadrant,,,,,,,
Q1 Fix now (AM-owned),32,"963,307.55",1477,1295,78.02,0.09,0.20
Q2 Protect (keep happy),167,"6,729,495.07",2705,5335,41.82,0.62,0.37
Q3 Automate nudges / restrict,119,"698,476.07",1554,1415,79.72,0.06,0.21
Q4 Self-serve,477,"2,485,376.32",1612,2794,43.67,0.23,0.22


## 4. Back-test

Here I build the score using only Jan 2017 - Mar 2018 data, and then compare it with how the same sellers performed in Apr-Aug 2018. If the score is useful, higher risk groups should have more late handoffs and bad reviews later.

In [5]:
train_o, test_o = d[d.purchase_month <= "2018-03"], d[d.purchase_month >= "2018-04"]
train_c = canc[canc.purchase_month <= "2018-03"]
past = score_sellers(train_o, train_c)
fut = test_o.groupby("seller_id").agg(fut_orders=("order_id", "count"),
                                      fut_handoff=("seller_late_handoff", "mean"),
                                      fut_bad=("review_score", lambda s: (s <= 2).mean()))
bt = past.join(fut, how="inner")
bt = bt[bt.fut_orders >= 5]
bt["risk_band"] = pd.qcut(bt.risk_score, 5, labels=["Lowest", "Low", "Mid", "High", "Highest"])
band = bt.groupby("risk_band", observed=True).apply(lambda t: pd.Series({
    "sellers": len(t), "future_orders": t.fut_orders.sum(),
    "future_late_handoff_pct": np.average(t.fut_handoff, weights=t.fut_orders) * 100,
    "future_bad_review_pct": np.average(t.fut_bad, weights=t.fut_orders) * 100}))
from scipy.stats import spearmanr
rho = spearmanr(bt.risk_score, bt.fut_handoff).statistic
print(f"Sellers in back-test: {len(bt):,} | Spearman(risk score, future late-handoff rate) = {rho:.2f}")
band.round(2)

Sellers in back-test: 428 | Spearman(risk score, future late-handoff rate) = 0.28


,sellers,future_orders,future_late_handoff_pct,future_bad_review_pct
risk_band,,,,
Lowest,86.00,"4,377.00",3.54,8.06
Low,85.00,"3,192.00",4.92,8.11
Mid,86.00,"4,781.00",4.75,10.21
High,85.00,"3,479.00",6.29,11.56
Highest,86.00,"4,066.00",15.27,13.99


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"wspace": 0.25})
seq = ["#fdd0a2", "#fdae6b", "#fd8d3c", "#e6550d", "#a63603"]
for ax, col, ttl in [(axes[0], "future_late_handoff_pct", "Future late-handoff rate (%)"),
                     (axes[1], "future_bad_review_pct", "Future 1-2 star review rate (%)")]:
    ax.bar(band.index.astype(str), band[col], color=seq, width=0.6)
    for i, v in enumerate(band[col]): ax.text(i, v + 0.2, f"{v:.1f}%", ha="center", fontsize=9, color=cs.GREY)
    ax.set_xlabel("Risk quintile (scored on data up to Mar-2018)"); ax.set_ylabel(ttl); ax.grid(axis="x", visible=False)
fig.suptitle("Back-test: Apr-Aug 2018 performance by risk quintile", fontweight="bold", fontsize=12)
r0, r4 = band.future_late_handoff_pct.iloc[0], band.future_late_handoff_pct.iloc[-1]
cs.save(fig, "p4_backtest")

PosixPath('/home/claude/olist-ops-growth-case-study/images/p4_backtest.png')

## 5. Export for Excel

In [7]:
sellers = pd.read_csv(OUT / "seller_scorecard.csv").set_index("seller_id")[["seller_state", "seller_city"]]
out = (sc.join(sellers).reset_index()
         .assign(tier=lambda t: pd.cut(t.risk_score, [0, 50, 70, 85, 100], labels=["Healthy", "Watch", "High", "Critical"], include_lowest=True))
         .sort_values(["quadrant", "risk_score"], ascending=[True, False]))
cols = ["seller_id", "seller_state", "seller_city", "quadrant", "tier", "risk_score", "delivered", "gmv",
        "handoff_rate", "bad_review_rate", "cancel_rate", "late_handoff", "bad", "canceled"]
out[cols].round(4).to_csv(OUT / "seller_risk_tiers.csv", index=False)
print(f"{len(out):,} sellers exported | Q1 'Fix now' list: {(out.quadrant.str.startswith('Q1')).sum()} sellers")
out[cols].head(8).round(3)

795 sellers exported | Q1 'Fix now' list: 32 sellers


,seller_id,seller_state,seller_city,quadrant,tier,risk_score,delivered,gmv,handoff_rate,bad_review_rate,cancel_rate,late_handoff,bad,canceled
437,8444e55c1f13cd5c179851e5ca5ebd00,MG,congonhal,Q1 Fix now (AM-owned),Critical,96.03,91,"21,464.56",0.21,0.28,0.04,22,29,4
447,88460e8ebdecbfecb5f9601833981930,PR,maringa,Q1 Fix now (AM-owned),Critical,95.82,232,"30,769.90",0.47,0.28,0.00,116,67,1
440,855668e0971d4dfd7bef1b6a4133b41b,SP,itatiba,Q1 Fix now (AM-owned),Critical,89.23,297,"30,703.22",0.18,0.20,0.01,55,59,2
241,4a3ccda38b2129705f3fb522db62ca31,SP,marilia,Q1 Fix now (AM-owned),High,83.67,62,"15,898.29",0.13,0.17,0.01,9,11,1
525,a673821011d0cec28146ea42f5ab767f,SP,sao paulo,Q1 Fix now (AM-owned),High,83.50,122,"13,218.04",0.16,0.16,0.01,21,19,2
633,cd68562d3f44870c08922d380acae552,SP,ribeirao preto,Q1 Fix now (AM-owned),High,82.29,122,"18,119.70",0.14,0.16,0.01,18,20,1
467,8e6d7754bc7e0f22c96d255ebda59eba,SP,sao jose dos campos,Q1 Fix now (AM-owned),High,81.55,82,"15,027.08",0.32,0.23,0.00,31,21,0
726,e9bc59e7b60fc3063eb2290deda4cced,PR,maringa,Q1 Fix now (AM-owned),High,80.48,203,"33,058.04",0.25,0.14,0.01,54,28,1


## What to do with each group

| Group | Sellers | Action | How often | KPI |
|---|---|---|---|---|
| Q1 Fix now | High GMV, high risk | Assign an account manager; find the cause (stock, staff, carrier pickup); 30-day plan | Weekly | Late handoff < 5% within 60 days |
| Q2 Protect | High GMV, low risk | Quarterly review, early access to promotions | Quarterly | GMV growth, retention |
| Q3 Automate / restrict | Low GMV, high risk | Automatic emails with their own numbers, ranking penalty, suspension after 3 strikes | Automatic | Risk score trend |
| Q4 Self-serve | Low GMV, low risk | Help centre only | - | - |

Q1 has only 32 sellers but they cause ~20% of all late handoffs. With 3 account managers that's about 11 sellers each. These numbers can be changed in the `Capacity_Model` sheet in Excel.